In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned data (not raw!)
df = pd.read_csv("04_Featureengg .csv")
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male,BalanceToSalary,ProductsPerYear,EngagementScore,AgeWhenJoined
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False,0.000000,0.333333,1,40
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False,0.744670,0.500000,1,40
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False,1.401362,0.333333,0,34
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False,0.000000,1.000000,0,38
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False,1.587035,0.333333,1,41


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1 — Separate features and target
X = df.drop("Exited", axis=1)
y = df["Exited"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print(X.columns.tolist())

# Step 2 — Split first
X_train, X_test, y_train, y_test = train_test_split(
                                        X, y,
                                        test_size=0.2,
                                        random_state=42,
                                        stratify=y)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

# Step 3 — Scale AFTER split
scaler = StandardScaler()

num_cols = ["CreditScore", "Age", "Tenure", "Balance",
            "NumOfProducts", "EstimatedSalary",
            "BalanceToSalary", "ProductsPerYear",
            "EngagementScore", "AgeWhenJoined"]

# Fit only on train, transform both
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

# Verify
print("\nAfter Scaling:")
print("y_train churn rate:", round(y_train.mean() * 100, 2), "%")
print("y_test churn rate: ", round(y_test.mean()  * 100, 2), "%")
print(X_train.head())

X shape: (9988, 15)
y shape: (9988,)
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Gender_Male', 'BalanceToSalary', 'ProductsPerYear', 'EngagementScore', 'AgeWhenJoined']
X_train: (7990, 15)
X_test:  (1998, 15)

After Scaling:
y_train churn rate: 20.39 %
y_test churn rate:  20.37 %
      CreditScore       Age    Tenure   Balance  NumOfProducts  HasCrCard  \
3539     0.838045 -0.562626 -1.399820 -1.236191      -0.912086          1   
1883     0.062850 -0.368793  0.329395 -1.236191       0.809929          1   
8675     1.261818  0.406541  0.329395 -1.236191       0.809929          1   
6253    -0.671001 -1.047210  1.366924  0.939502       0.809929          1   
4131     0.011170  0.503457 -1.053977 -1.236191       2.531943          1   

      IsActiveMember  EstimatedSalary  Geography_Germany  Geography_Spain  \
3539               0         0.524526              False             Tr

In [6]:
bool_cols = ["Geography_Germany", "Geography_Spain", "Gender_Male"]
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols]  = X_test[bool_cols].astype(int)

print(X_train.head())

      CreditScore       Age    Tenure   Balance  NumOfProducts  HasCrCard  \
3539     0.838045 -0.562626 -1.399820 -1.236191      -0.912086          1   
1883     0.062850 -0.368793  0.329395 -1.236191       0.809929          1   
8675     1.261818  0.406541  0.329395 -1.236191       0.809929          1   
6253    -0.671001 -1.047210  1.366924  0.939502       0.809929          1   
4131     0.011170  0.503457 -1.053977 -1.236191       2.531943          1   

      IsActiveMember  EstimatedSalary  Geography_Germany  Geography_Spain  \
3539               0         0.524526                  0                1   
1883               0        -1.725057                  0                0   
8675               1        -0.747003                  0                0   
6253               1        -1.047957                  1                0   
4131               0         0.033735                  0                0   

      Gender_Male  BalanceToSalary  ProductsPerYear  EngagementScore  \
35

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Define model
model = LogisticRegression(random_state=42)

# Stratified K-Fold (preserves churn ratio in each fold)
skf = StratifiedKFold(n_splits=5,    # 5 folds
                      shuffle=True, 
                      random_state=42)

# Run cross validation
scores = cross_val_score(model, 
                         X_train, y_train,
                         cv=skf,
                         scoring="accuracy")

print("Score each fold:", scores)
print("Mean accuracy: ", scores.mean() * 100)
print("Std deviation: ", scores.std() * 100)

Score each fold: [0.80475594 0.81476846 0.79849812 0.81539424 0.81038798]
Mean accuracy:  80.87609511889863
Std deviation:  0.6389110897672184


In [12]:
# Save cleaned dataframe as CSV
df.to_csv("05_Train_test&Scaling .csv", index=False)
print("Saved successfully!")

Saved successfully!


In [15]:
import pickle

with open("05_Train_test&Scaling .pkl", "wb") as f:
    pickle.dump((X_train, X_test, y_train, y_test), f)

print("Saved!")

Saved!


In [16]:
import pickle

# Save scaler separately
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Scaler saved!")

Scaler saved!


In [17]:
print(scaler.feature_names_in_)
print(len(scaler.feature_names_in_))

['CreditScore' 'Age' 'Tenure' 'Balance' 'NumOfProducts' 'EstimatedSalary'
 'BalanceToSalary' 'ProductsPerYear' 'EngagementScore' 'AgeWhenJoined']
10
